In [3]:
## %pip install mujoco

  Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 65.7 MB/s eta 0:00:00a 0:00:01
Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [mujoco]2m7/8 [mujoco]]
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import Standard libraries
from pathlib import Path
import time
import numpy as np

#Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [27]:
# Settings
MJCF_PATH = Path("unitree_go2/scene_indoor.xml")

CONTROL_STEP = 0.002   # seconds per control update
PRINT_EVERY = 50

if not MJCF_PATH.is_file():
    raise FileNotFoundError(f"Could not find: {MJCF_PATH.resolve()}")

model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)

print("Loaded:", MJCF_PATH.resolve())
print("Actuators:", model.nu)
print("Sensors:", model.nsensor)

Loaded: /mnt/c/users/Raged Rhombus/Desktop/Projects/Go2-Learning/unitree_go2/scene_indoor.xml
Actuators: 12
Sensors: 30


In [22]:
## help(mujoco.mjtObj) ## To show the structure of the "object" of our model

In [10]:
ACTUATOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    for i in range(model.nu)
]

SENSOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_SENSOR, i)
    for i in range(model.nsensor)
]

print("Actuators:")
for i, name in enumerate(ACTUATOR_NAMES):
    print(i, name)

print("\nSensors:")
for i, name in enumerate(SENSOR_NAMES):
    print(i, name)

Actuators:
0 FL_hip
1 FL_thigh
2 FL_calf
3 FR_hip
4 FR_thigh
5 FR_calf
6 RL_hip
7 RL_thigh
8 RL_calf
9 RR_hip
10 RR_thigh
11 RR_calf

Sensors:
0 abduction_front_left_pos
1 hip_front_left_pos
2 knee_front_left_pos
3 abduction_hind_left_pos
4 hip_hind_left_pos
5 knee_hind_left_pos
6 abduction_front_right_pos
7 hip_front_right_pos
8 knee_front_right_pos
9 abduction_hind_right_pos
10 hip_hind_right_pos
11 knee_hind_right_pos
12 abduction_front_left_vel
13 hip_front_left_vel
14 knee_front_left_vel
15 abduction_hind_left_vel
16 hip_hind_left_vel
17 knee_hind_left_vel
18 abduction_front_right_vel
19 hip_front_right_vel
20 knee_front_right_vel
21 abduction_hind_right_vel
22 hip_hind_right_vel
23 knee_hind_right_vel
24 gyro
25 accelerometer
26 orientation
27 global_position
28 global_linvel
29 global_angvel


In [14]:
# control limits defined by the model

for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "limited =", bool(model.actuator_ctrllimited[i]),
        "range =", model.actuator_ctrlrange[i]
    )

FL_hip limited = True range = [-0.9472  0.9472]
FL_thigh limited = True range = [-1.4  2.5]
FL_calf limited = True range = [-2.6227  -0.84776]
FR_hip limited = True range = [-0.9472  0.9472]
FR_thigh limited = True range = [-1.4  2.5]
FR_calf limited = True range = [-2.6227  -0.84776]
RL_hip limited = True range = [-0.9472  0.9472]
RL_thigh limited = True range = [-1.4  2.5]
RL_calf limited = True range = [-2.6227  -0.84776]
RR_hip limited = True range = [-0.9472  0.9472]
RR_thigh limited = True range = [-1.4  2.5]
RR_calf limited = True range = [-2.6227  -0.84776]


In [15]:
for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "transmission:", model.actuator_trntype[i],
        "dynamics:", model.actuator_dyntype[i],
        "gain:", model.actuator_gainprm[i],
        "bias:", model.actuator_biasprm[i],
        "range:", model.actuator_ctrlrange[i],
    )

FL_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FL_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FL_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-2.6227  -0.84776]
FR_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FR_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FR_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.

In [18]:
ACTUATOR_TO_POSITION_SENSOR = {
    "FL_hip":   "abduction_front_left_pos",
    "FL_thigh": "hip_front_left_pos",
    "FL_calf":  "knee_front_left_pos",

    "FR_hip":   "abduction_front_right_pos",
    "FR_thigh": "hip_front_right_pos",
    "FR_calf":  "knee_front_right_pos",

    "RL_hip":   "abduction_hind_left_pos",
    "RL_thigh": "hip_hind_left_pos",
    "RL_calf":  "knee_hind_left_pos",

    "RR_hip":   "abduction_hind_right_pos",
    "RR_thigh": "hip_hind_right_pos",
    "RR_calf":  "knee_hind_right_pos",
}

In [30]:
home_id = mujoco.mj_name2id(
    model, mujoco.mjtObj.mjOBJ_KEY, "home"
)

mujoco.mj_resetDataKeyframe(model, data, home_id)
mujoco.mj_forward(model, data)

home_ctrl = data.ctrl.copy()

In [31]:
# Separate data used only for calculating joint angles.
# The simulated robot remains controlled through data.ctrl.
ik_data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, ik_data, home_id)
mujoco.mj_forward(model, ik_data)

legs = {}
for leg in ("FL", "FR", "RL", "RR"):
    names = [f"{leg}_{part}" for part in ("hip", "thigh", "calf")]
    joint_ids = [model.joint(name + "_joint").id for name in names]
    actuator_ids = np.array([model.actuator(name).id for name in names])
    site_id = model.site(f"{leg}_foot").id

    legs[leg] = {
        "site": site_id,
        "qpos": model.jnt_qposadr[joint_ids],
        "dof": model.jnt_dofadr[joint_ids],
        "act": actuator_ids,
        "foot_home": ik_data.site_xpos[site_id].copy(),
        "limits": model.actuator_ctrlrange[actuator_ids].copy(),
    }

In [32]:
PERIOD = 0.8          # Seconds per complete gait cycle
STEP_LENGTH = 0.04   # 4 cm of front-to-back foot travel
STEP_HEIGHT = 0.025  # 2.5 cm lift
STANCE_FRACTION = 0.65  # Overlap gives time with all four feet down

# Diagonal pairs share the same phase.
PHASE = {"FL": 0.0, "RR": 0.0, "FR": 0.5, "RL": 0.5}


def foot_offset(phase):
    if phase < STANCE_FRACTION:
        s = phase / STANCE_FRACTION
        x = STEP_LENGTH * (0.5 - s)
        z = 0.0
    else:
        s = (phase - STANCE_FRACTION) / (1 - STANCE_FRACTION)
        smooth = 3 * s**2 - 2 * s**3
        x = STEP_LENGTH * (smooth - 0.5)
        z = STEP_HEIGHT * np.sin(np.pi * s)**2

    return np.array([x, 0.0, z])


def solve_leg(leg, target):
    """Find joint angles for a foot target in the fixed IK reference frame."""
    info = legs[leg]
    jacobian = np.zeros((3, model.nv))

    for _ in range(15):
        mujoco.mj_forward(model, ik_data)
        error = target - ik_data.site_xpos[info["site"]]

        if np.linalg.norm(error) < 0.0005:
            break

        mujoco.mj_jacSite(
            model, ik_data, jacobian, None, info["site"]
        )
        J = jacobian[:, info["dof"]]

        # Damped inverse: convert a foot-position error into joint changes.
        dq = J.T @ np.linalg.solve(
            J @ J.T + 1e-4 * np.eye(3), error
        )

        ik_data.qpos[info["qpos"]] = np.clip(
            ik_data.qpos[info["qpos"]] + np.clip(dq, -0.1, 0.1),
            info["limits"][:, 0],
            info["limits"][:, 1],
        )

    mujoco.mj_forward(model, ik_data)
    residual = np.linalg.norm(target - ik_data.site_xpos[info["site"]])
    if residual > 0.003:
        raise RuntimeError(f"{leg}: foot target missed by {residual:.3f} m")

    return ik_data.qpos[info["qpos"]].copy()


def gait_targets(t):
    targets = home_ctrl.copy()
    blend = min(t / 1.5, 1.0)  # Gradually introduce stepping.

    for leg, info in legs.items():
        phase = (t / PERIOD + PHASE[leg]) % 1.0
        target = info["foot_home"] + blend * foot_offset(phase)
        targets[info["act"]] = solve_leg(leg, target)

    return targets

In [33]:
mujoco.mj_resetDataKeyframe(model, data, home_id)
mujoco.mj_resetDataKeyframe(model, ik_data, home_id)
mujoco.mj_forward(model, data)

start_x = float(data.body("base").xpos[0])
next_report = 1.0
next_control = 0.0
dt = model.opt.timestep

with mujoco.viewer.launch_passive(model, data) as viewer:
    with viewer.lock():
        viewer.cam.distance = 1.5
        viewer.cam.azimuth = 135
        viewer.cam.elevation = -20
        viewer.cam.lookat[:] = data.body("base").xpos

    while viewer.is_running() and data.time < 8.0:
        wall_start = time.perf_counter()

        if data.time >= next_control:
            if data.time < 2.0:
                data.ctrl[:] = home_ctrl
            else:
                data.ctrl[:] = gait_targets(data.time - 2.0)

            next_control = data.time + 0.01  # Approximately 100 Hz

        mujoco.mj_step(model, data)
        mujoco.mj_forward(model, data)

        base = data.body("base")
        height = float(base.xpos[2])
        upright = base.xmat.reshape(3, 3)[2, 2]

        viewer.sync()

        if height < 0.15 or upright < np.cos(np.deg2rad(35)):
            print("Stopped: body dropped or tilted too far.")
            break

        if data.time >= next_report:
            dx = float(base.xpos[0]) - start_x
            vx = float(data.sensor("global_linvel").data[0])
            print(
                f"t={data.time:.1f}s | "
                f"forward distance={dx:.3f}m | speed={vx:.3f}m/s"
            )
            next_report += 1.0

        time.sleep(max(0.0, dt - (time.perf_counter() - wall_start)))

t=1.0s | forward distance=-0.033m | speed=-0.009m/s
t=2.0s | forward distance=-0.036m | speed=-0.001m/s
t=3.0s | forward distance=-0.038m | speed=0.029m/s
t=4.0s | forward distance=-0.041m | speed=-0.066m/s
t=5.0s | forward distance=-0.046m | speed=0.050m/s
t=6.0s | forward distance=-0.048m | speed=-0.067m/s
t=7.0s | forward distance=-0.052m | speed=0.050m/s
t=8.0s | forward distance=-0.054m | speed=-0.066m/s
